In [ ]:
import os
import sys
import pickle
import logging
import datetime
import pickle
import platform
computername = platform.node()
import pathlib
from collections import deque
from dataclasses import dataclass, field
from IPython.display import display, HTML

In [ ]:
search_dirs = ['../..', '../../..', '..']
pkg_needed = ['eventkit', 'ib_insync']
pkg_dirs = {}

for pkg in pkg_needed:
    for dir in search_dirs:
        potential_dir = pathlib.Path(dir, pkg).resolve()
        if potential_dir.exists() and potential_dir.is_dir():
            pkg_dirs[pkg] = potential_dir
            break

for pkg, pkg_dir in pkg_dirs.items():
    if str(pkg_dir) not in sys.path:
        sys.path.insert(0, str(pkg_dir))
        print(f"{pkg_dir} added to sys.path")

missing_pkgs = [pkg for pkg in pkg_needed if pkg not in pkg_dirs]
if missing_pkgs:
    print(f"Expected directories not found for: {', '.join(missing_pkgs)}")


In [ ]:
import ib_insync
import ib_insync.util as util
ib_insync.ib.install_custom_repr_()


In [ ]:
logging.basicConfig(level=logging.INFO
    , format='%(asctime)s - %(name)s - %(levelname)s - %(funcName)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
# program_name = os.path.splitext(os.path.basename(__file__))[0] 
scriptdir = os.getcwd()
logger.info(f"Script directory: {scriptdir}")
datafilesuffixdt = f'AAPL_{datetime.datetime.now():%y%m%d}_{computername}'

pklfile = os.path.join(scriptdir, 'data', f'tracker_{datafilesuffixdt}.pkl')
fpkl = None
data = None
if not os.path.exists(pklfile) or os.path.getsize(pklfile) == 0:
    logger.info(f"Initialized pkl file: {pklfile}")
else:
    fpkl = open(pklfile, 'r+b') # read/write binary
    data = pickle.load(fpkl)
    logger.info(f"Previous session data: {display(data)}")


In [ ]:
def custom_formatter(value):
    if isinstance(value, (bool, int, float, str, bytes)):
        return str(value)
    if isinstance(value, datetime.datetime):
        value_local = value.astimezone()  # Convert to local time
        if value.date() == datetime.datetime.now().date():
            return value_local.strftime('%H:%M:%S')
        else:
            return value_local.strftime(r'%Y-%m-%d %H:%M:%S %Z')
    elif isinstance(value, list):
        return '[' + ", ".join([custom_formatter(v) for v in value]) + ']'
    elif isinstance(value, dict):
        return {k: custom_formatter(v) for k, v in value.items()}
    # elif util.isnamedtupleinstance(value):
    #     return {f: custom_formatter(getattr(value, f)) for f in value._fields}
    # elif util.is_dataclass(value):
    #     return {value.__class__.__qualname__: custom_formatter(util.dataclassNonDefaults(value))}
    else:
        # logger.warning(f"Unknown type: {type(value)}")
        return str(value)

def print_dict(data):
    for key, value in data.items():
        formatted_value = custom_formatter(value)
        if isinstance(value, datetime.datetime):
            print(f"{key}: {str(formatted_value)}")
        else:
            print(f"{key}: {formatted_value}")

def repr_dict(data):
    return {key: custom_formatter(value) for key, value in data.items()}

def format_dict(d: dict) -> str:
    return '{' + ", ".join(f"{k}: {custom_formatter(v)}" for k, v in d.items()) + '}'

# my_dict = {"name": "Alice", "age": 30}
# print("My dict: {my_dict}".format(my_dict=format_dict(my_dict)))

# Example usage
print(format_dict(data))
#print_dict(data)

In [ ]:
@dataclass
class A:
    volatility_per_min: deque = field(default_factory=deque) # volatility every minute


In [ ]:
a = A()

In [ ]:
if not a.volatility_per_min:
    print("Empty")